In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import pandas as pd
import torch
import re

In [ ]:
model_name = "facebook/nllb-200-3.3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")

In [ ]:
input_file = "path/to/input.tsv"
df = pd.read_csv(input_file, sep='\t')

In [ ]:
src_lang = "eng_Latn"
tgt_lang = "ita_Latn"

def translate(texts):
    input_texts = [f"Robot, {t}" for t in texts]

    tokenizer.src_lang = src_lang
    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        length_penalty=1.0,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    translations = []
    for out in outputs:
        text = tokenizer.decode(out, skip_special_tokens=True)
        try:
            text = text.removeprefix("Robot, ").lstrip()
            text = re.sub(r"[^\w\s']", '', text, flags=re.UNICODE).lower()
            text = re.sub(r"(\w)'\s*(\w)", r"\1' \2", text).strip()
        except:
            text = "TRANSLATION ERROR"
        translations.append(text)

    return translations

In [ ]:
def translate_dataframe(df, batch_size=10):
    results = []
    for i in tqdm(range(0, len(df), batch_size), desc="Processing batch"):
        batch_df = df.iloc[i:i+batch_size]
        batch_texts = batch_df['input'].astype(str).tolist()
        batch_ids = batch_df['id'].tolist()

        batch_translations = translate(batch_texts)

        results.extend(list(zip(batch_ids, batch_texts, batch_translations)))
    
    return results

In [ ]:
translations = translate_dataframe(df)

In [ ]:
df_result = pd.DataFrame(translations, columns=["id", "input", "translation"])
output_file = "path/to/output/nllb200.tsv"
df_result.to_csv(output_file, sep='\t', index=False)
print(f"Translation completed. File saved in: {output_file}")